# Imports y SparkSession

In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, upper, lower, when, lit,
    count, countDistinct, avg, sum as spark_sum,
    round, desc, first, coalesce,
    to_timestamp, to_date, date_format,
    year, month, concat_ws, regexp_replace,
    add_months, max as spark_max
)
from pyspark.sql.window import Window
from pyspark.sql.functions import lag

In [3]:
spark = (
    SparkSession.builder
    .appName("Proyecto Big Data - Restaurantes NYC - Spark")
    .getOrCreate()
)

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/05 16:41:54 INFO SparkEnv: Registering MapOutputTracker
26/05/05 16:41:54 INFO SparkEnv: Registering BlockManagerMaster
26/05/05 16:41:54 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/05/05 16:41:55 INFO SparkEnv: Registering OutputCommitCoordinator


In [4]:
GCS_RAW_PATH = "gs://proyectobigdata2026/raw/restaurant_inspections.csv"

GCS_PROCESSED_SPARK = "gs://proyectobigdata2026/processed/spark/restaurant_inspections_clean/"

GCS_OUTPUTS_SPARK = "gs://proyectobigdata2026/results/outputs/spark/"
GCS_KPIS_SPARK = "gs://proyectobigdata2026/results/kpis/spark/"

In [5]:
!gcloud storage ls gs://proyectobigdata2026/raw/

gs://proyectobigdata2026/raw/

gs://proyectobigdata2026/raw/:
gs://proyectobigdata2026/raw/
gs://proyectobigdata2026/raw/restaurant_inspections.csv


In [6]:
df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("multiLine", False)
    .option("escape", '"')
    .csv(GCS_RAW_PATH)
)

df_raw.show(5, truncate=False)
df_raw.printSchema()

print("Filas originales:", df_raw.count())
print("Columnas:", len(df_raw.columns))

26/05/05 16:42:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------+-----------------+---------+--------+--------------+-------+----------+-------------------+-----------------------+-----------------------------------------------+--------------+------------------------------------------------------------------------+--------------+-----+-----+----------+-----------------------+-------------------------------------------------+----------------+---------------+----------------+-------+---------------+----+------------+----------+----------------------------------------+
|camis   |dba              |boro     |building|street        |zipcode|phone     |cuisine_description|inspection_date        |action                                         |violation_code|violation_description                                                   |critical_flag |score|grade|grade_date|record_date            |inspection_type                                  |longitude       |latitude       |council_district|bin    |community_board|nta |census_tract|bbl       |locati

Filas originales: 296740
Columnas: 27


# Tratamiento de datos

In [7]:
def normalize_column_name(name):
    return (
        name.strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

df = df_raw.toDF(*[normalize_column_name(c) for c in df_raw.columns])

df.printSchema()
df.show(5, truncate=False)

root
 |-- camis: string (nullable = true)
 |-- dba: string (nullable = true)
 |-- boro: string (nullable = true)
 |-- building: string (nullable = true)
 |-- street: string (nullable = true)
 |-- zipcode: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- cuisine_description: string (nullable = true)
 |-- inspection_date: string (nullable = true)
 |-- action: string (nullable = true)
 |-- violation_code: string (nullable = true)
 |-- violation_description: string (nullable = true)
 |-- critical_flag: string (nullable = true)
 |-- score: string (nullable = true)
 |-- grade: string (nullable = true)
 |-- grade_date: string (nullable = true)
 |-- record_date: string (nullable = true)
 |-- inspection_type: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- council_district: string (nullable = true)
 |-- bin: string (nullable = true)
 |-- community_board: string (nullable = true)
 |-- nta: string (nullable = true)
 

In [8]:
selected_columns = [
    "camis",
    "dba",
    "boro",
    "building",
    "street",
    "zipcode",
    "phone",
    "cuisine_description",
    "inspection_date",
    "action",
    "violation_code",
    "violation_description",
    "critical_flag",
    "score",
    "grade",
    "grade_date",
    "record_date",
    "inspection_type",
    "latitude",
    "longitude",
    "community_board",
    "council_district",
    "census_tract",
    "bin",
    "bbl",
    "nta",
    "location"
]

existing_columns = [c for c in selected_columns if c in df.columns]
df = df.select(*existing_columns)

df.show(5, truncate=False)

+--------+-----------------+---------+--------+--------------+-------+----------+-------------------+-----------------------+-----------------------------------------------+--------------+------------------------------------------------------------------------+--------------+-----+-----+----------+-----------------------+-------------------------------------------------+---------------+----------------+---------------+----------------+------------+-------+----------+----+----------------------------------------+
|camis   |dba              |boro     |building|street        |zipcode|phone     |cuisine_description|inspection_date        |action                                         |violation_code|violation_description                                                   |critical_flag |score|grade|grade_date|record_date            |inspection_type                                  |latitude       |longitude       |community_board|council_district|census_tract|bin    |bbl       |nta |locati

### Perfilamiento inicial

In [9]:
null_summary = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in ["camis", "inspection_date", "score", "boro", "critical_flag", "violation_code"]
])

null_summary.show(truncate=False)

+-----+---------------+-----+----+-------------+--------------+
|camis|inspection_date|score|boro|critical_flag|violation_code|
+-----+---------------+-----+----+-------------+--------------+
|0    |0              |17061|0   |0            |5833          |
+-----+---------------+-----+----+-------------+--------------+



In [10]:
df.groupBy("boro").count().orderBy(desc("count")).show(truncate=False)
df.groupBy("grade").count().orderBy(desc("count")).show(truncate=False)
df.groupBy("critical_flag").count().orderBy(desc("count")).show(truncate=False)

+-------------+------+
|boro         |count |
+-------------+------+
|Manhattan    |110189|
|Brooklyn     |75544 |
|Queens       |73748 |
|Bronx        |27118 |
|Staten Island|9811  |
|0            |330   |
+-------------+------+



+-----+------+
|grade|count |
+-----+------+
|NULL |150586|
|A    |99277 |
|B    |18577 |
|C    |13578 |
|N    |9545  |
|Z    |4300  |
|P    |877   |
+-----+------+



+--------------+------+
|critical_flag |count |
+--------------+------+
|Critical      |156073|
|Not Critical  |132930|
|Not Applicable|7737  |
+--------------+------+



# Limpieza base

In [11]:
text_columns = [
    "camis", "dba", "boro", "building", "street", "zipcode",
    "phone", "cuisine_description", "action", "violation_code",
    "violation_description", "critical_flag", "grade",
    "inspection_type", "nta"
]

for c in text_columns:
    if c in df.columns:
        df = df.withColumn(c, trim(col(c).cast("string")))

In [12]:
df_clean = (
    df
    .withColumn("dba", upper(col("dba")))
    .withColumn("grade", upper(col("grade")))
    .withColumn("critical_flag", trim(col("critical_flag")))
)

In [13]:
df_clean = (
    df_clean
    .withColumn(
        "inspection_ts",
        coalesce(
            to_timestamp(col("inspection_date"), "yyyy-MM-dd'T'HH:mm:ss.SSS"),
            to_timestamp(col("inspection_date"), "yyyy-MM-dd'T'HH:mm:ss"),
            to_timestamp(col("inspection_date"), "MM/dd/yyyy")
        )
    )
    .withColumn(
        "grade_ts",
        coalesce(
            to_timestamp(col("grade_date"), "yyyy-MM-dd'T'HH:mm:ss.SSS"),
            to_timestamp(col("grade_date"), "yyyy-MM-dd'T'HH:mm:ss"),
            to_timestamp(col("grade_date"), "MM/dd/yyyy")
        )
    )
    .withColumn(
        "record_ts",
        coalesce(
            to_timestamp(col("record_date"), "yyyy-MM-dd'T'HH:mm:ss.SSS"),
            to_timestamp(col("record_date"), "yyyy-MM-dd'T'HH:mm:ss"),
            to_timestamp(col("record_date"), "MM/dd/yyyy")
        )
    )
)

In [14]:
df_clean = (
    df_clean
    .drop("inspection_date", "grade_date", "record_date")
    .withColumnRenamed("inspection_ts", "inspection_date")
    .withColumnRenamed("grade_ts", "grade_date")
    .withColumnRenamed("record_ts", "record_date")
)

In [15]:
df_clean = (
    df_clean
    .withColumn("score", col("score").cast("double"))
    .withColumn("latitude", col("latitude").cast("double"))
    .withColumn("longitude", col("longitude").cast("double"))
)

In [16]:
df_clean = df_clean.filter(
    col("camis").isNotNull() &
    col("inspection_date").isNotNull() &
    (to_date(col("inspection_date")) != lit("1900-01-01"))
)

## Variables auxiliares

In [17]:
df_clean = (
    df_clean
    .withColumn("year", year(col("inspection_date")))
    .withColumn("month", month(col("inspection_date")))
    .withColumn(
        "season",
        when(col("month").isin(6, 7, 8), "Verano")
        .when(col("month").isin(12, 1, 2), "Invierno")
        .otherwise("Otra temporada")
    )
    .withColumn(
        "is_critical",
        when(lower(col("critical_flag")) == "critical", 1).otherwise(0)
    )
    .withColumn(
        "has_violation",
        when(col("violation_code").isNotNull(), 1).otherwise(0)
    )
    .withColumn(
        "inspection_id",
        concat_ws(
            "_",
            col("camis"),
            date_format(col("inspection_date"), "yyyy-MM-dd"),
            coalesce(col("inspection_type"), lit("unknown"))
        )
    )
)

In [18]:
df_clean = df_clean.withColumn(
    "risk_level",
    when(col("score").isNull(), "Sin puntaje")
    .when(col("score") <= 13, "Bajo")
    .when((col("score") >= 14) & (col("score") <= 27), "Medio")
    .otherwise("Alto")
)

In [19]:
df_clean = df_clean.withColumn(
    "sanitary_status",
    when((col("grade") == "A") & (col("is_critical") == 0), "Excelente")
    .when((col("grade") == "A") & (col("is_critical") == 1), "Aceptable con alerta")
    .when(col("grade").isin("B", "C"), "Riesgo moderado")
    .otherwise("Requiere revisión")
)

#### Corrección de boro usando zipcode

In [20]:
valid_boro_map = (
    df_clean
    .filter(
        col("zipcode").isNotNull() &
        col("boro").isNotNull() &
        (~col("boro").isin("0", "Missing", "MISSING", "missing"))
    )
    .groupBy("zipcode")
    .agg(first("boro", ignorenulls=True).alias("boro_from_zip"))
)

In [21]:
df_clean = (
    df_clean
    .join(valid_boro_map, on="zipcode", how="left")
    .withColumn(
        "boro_corrected",
        when(
            col("boro").isNull() |
            col("boro").isin("0", "Missing", "MISSING", "missing"),
            col("boro_from_zip")
        ).otherwise(col("boro"))
    )
    .withColumn(
        "boro_corrected",
        coalesce(col("boro_corrected"), lit("Sin datos"))
    )
    .drop("boro_from_zip")
)

# Base final para análisis

In [22]:
df_analysis = df_clean.filter(
    col("boro_corrected").isNotNull() &
    (col("boro_corrected") != "Sin datos") &
    col("score").isNotNull() &
    col("inspection_date").isNotNull()
)

In [23]:
print("Filas limpias:", df_clean.count())
print("Filas válidas para análisis:", df_analysis.count())

df_analysis.select(
    "camis", "dba", "boro_corrected", "cuisine_description",
    "inspection_date", "score", "grade",
    "critical_flag", "risk_level", "sanitary_status"
).show(10, truncate=False)

Filas limpias: 293267


Filas válidas para análisis: 279493


+--------+-----------------------+--------------+-------------------+-------------------+-----+-----+-------------+----------+-----------------+
|camis   |dba                    |boro_corrected|cuisine_description|inspection_date    |score|grade|critical_flag|risk_level|sanitary_status  |
+--------+-----------------------+--------------+-------------------+-------------------+-----+-----+-------------+----------+-----------------+
|41476556|POPEYES                |Queens        |Chicken            |2025-02-28 00:00:00|17.0 |NULL |Critical     |Medio     |Requiere revisión|
|50088657|MINNIE'S BAR           |Brooklyn      |American           |2024-10-02 00:00:00|26.0 |NULL |Critical     |Medio     |Requiere revisión|
|50047825|L'AMICO, BACK BAR      |Manhattan     |American           |2024-06-13 00:00:00|37.0 |NULL |Critical     |Alto      |Requiere revisión|
|50181761|DRAVIDA                |Manhattan     |Indian             |2026-03-31 00:00:00|42.0 |NULL |Not Critical |Alto      |Requ

## Checkpoint Spark en Parquet

In [24]:
(
    df_analysis
    .write
    .mode("overwrite")
    .parquet(GCS_PROCESSED_SPARK)
)

In [25]:
!gcloud storage ls gs://proyectobigdata2026/processed/spark/restaurant_inspections_clean/

gs://proyectobigdata2026/processed/spark/restaurant_inspections_clean/

gs://proyectobigdata2026/processed/spark/restaurant_inspections_clean/:
gs://proyectobigdata2026/processed/spark/restaurant_inspections_clean/
gs://proyectobigdata2026/processed/spark/restaurant_inspections_clean/_SUCCESS
gs://proyectobigdata2026/processed/spark/restaurant_inspections_clean/part-00000-b568bf60-b1c2-4313-87ed-15d6f3dff0e3-c000.snappy.parquet
gs://proyectobigdata2026/processed/spark/restaurant_inspections_clean/part-00001-b568bf60-b1c2-4313-87ed-15d6f3dff0e3-c000.snappy.parquet


In [26]:
df_analysis = spark.read.parquet(GCS_PROCESSED_SPARK)
df_analysis.printSchema()
df_analysis.show(5, truncate=False)

root
 |-- zipcode: string (nullable = true)
 |-- camis: string (nullable = true)
 |-- dba: string (nullable = true)
 |-- boro: string (nullable = true)
 |-- building: string (nullable = true)
 |-- street: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- cuisine_description: string (nullable = true)
 |-- action: string (nullable = true)
 |-- violation_code: string (nullable = true)
 |-- violation_description: string (nullable = true)
 |-- critical_flag: string (nullable = true)
 |-- score: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- inspection_type: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- community_board: string (nullable = true)
 |-- council_district: string (nullable = true)
 |-- census_tract: string (nullable = true)
 |-- bin: string (nullable = true)
 |-- bbl: string (nullable = true)
 |-- nta: string (nullable = true)
 |-- location: string (nullable = true)
 |-- inspectio

+-------+--------+-----------------+---------+--------+-------------------+----------+-------------------+-----------------------------------------------+--------------+------------------------------------------------------------------------+-------------+-----+-----+-------------------------------------------------+---------------+----------------+---------------+----------------+------------+-------+----------+----+----------------------------------------+-------------------+----------+-------------------+----+-----+--------------+-----------+-------------+---------------------------------------------------------------------+----------+-----------------+--------------+
|zipcode|camis   |dba              |boro     |building|street             |phone     |cuisine_description|action                                         |violation_code|violation_description                                                   |critical_flag|score|grade|inspection_type                                  |la

# CRUD con Spark

## SPARK-C1: Create — resultado de inspección

In [27]:
df_create = df_analysis.withColumn(
    "inspection_result",
    when(col("grade") == "A", "Cumple adecuadamente")
    .when(col("grade") == "B", "Cumple parcialmente")
    .when(col("grade") == "C", "Alto riesgo")
    .otherwise("Sin clasificación")
)

crud_c1_inspection_result = df_create.select(
    "camis", "dba", "grade", "inspection_result"
)

crud_c1_inspection_result.show(10, truncate=False)

+--------+-----------------------+-----+--------------------+
|camis   |dba                    |grade|inspection_result   |
+--------+-----------------------+-----+--------------------+
|41476556|POPEYES                |NULL |Sin clasificación   |
|50088657|MINNIE'S BAR           |NULL |Sin clasificación   |
|50047825|L'AMICO, BACK BAR      |NULL |Sin clasificación   |
|50181761|DRAVIDA                |NULL |Sin clasificación   |
|50132762|MIKADO                 |NULL |Sin clasificación   |
|50005067|AMERICAN WHISKEY       |NULL |Sin clasificación   |
|41701789|FLACO'S PIZZERIA       |NULL |Sin clasificación   |
|50146853|CAFE LOUNGE            |NULL |Sin clasificación   |
|41616416|ISLAND CHOIZ RESTAURANT|B    |Cumple parcialmente |
|50138665|SAKE JAPANESE CUISINE  |A    |Cumple adecuadamente|
+--------+-----------------------+-----+--------------------+
only showing top 10 rows



## SPARK-R1: Read — resumen por distrito

In [28]:
crud_r1_boro_summary = (
    df_analysis.groupBy("boro_corrected")
    .agg(
        countDistinct("inspection_id").alias("total_inspections"),
        countDistinct("camis").alias("total_restaurants"),
        round(avg("score"), 2).alias("avg_score")
    )
    .orderBy(desc("total_inspections"))
)

crud_r1_boro_summary.show(truncate=False)

+--------------+-----------------+-----------------+---------+
|boro_corrected|total_inspections|total_restaurants|avg_score|
+--------------+-----------------+-----------------+---------+
|Manhattan     |31695            |10834            |24.57    |
|Brooklyn      |21760            |7029             |25.9     |
|Queens        |19508            |6288             |26.8     |
|Bronx         |7691             |2412             |23.79    |
|Staten Island |3001             |1006             |22.97    |
+--------------+-----------------+-----------------+---------+



## SPARK-U1: Update — prioridad de atención

In [29]:
df_update = df_analysis.withColumn(
    "priority_attention",
    when((col("risk_level") == "Alto") | (col("is_critical") == 1), "Alta prioridad")
    .when(col("risk_level") == "Medio", "Prioridad media")
    .otherwise("Baja prioridad")
)

crud_u1_priority_attention = df_update.select(
    "camis", "dba", "boro_corrected", "score",
    "risk_level", "critical_flag", "priority_attention"
)

crud_u1_priority_attention.show(10, truncate=False)

+--------+-----------------------+--------------+-----+----------+-------------+------------------+
|camis   |dba                    |boro_corrected|score|risk_level|critical_flag|priority_attention|
+--------+-----------------------+--------------+-----+----------+-------------+------------------+
|41476556|POPEYES                |Queens        |17.0 |Medio     |Critical     |Alta prioridad    |
|50088657|MINNIE'S BAR           |Brooklyn      |26.0 |Medio     |Critical     |Alta prioridad    |
|50047825|L'AMICO, BACK BAR      |Manhattan     |37.0 |Alto      |Critical     |Alta prioridad    |
|50181761|DRAVIDA                |Manhattan     |42.0 |Alto      |Not Critical |Alta prioridad    |
|50132762|MIKADO                 |Queens        |25.0 |Medio     |Critical     |Alta prioridad    |
|50005067|AMERICAN WHISKEY       |Manhattan     |28.0 |Alto      |Critical     |Alta prioridad    |
|41701789|FLACO'S PIZZERIA       |Manhattan     |30.0 |Alto      |Critical     |Alta prioridad    |


## SPARK-D1: Delete lógico — resumen de filtrado

In [30]:
original_count = df_raw.count()
clean_count = df_clean.count()
analysis_count = df_analysis.count()

crud_d1_delete_summary = spark.createDataFrame(
    [
        ("dataset_original", original_count, 0),
        ("dataset_clean", clean_count, original_count - clean_count),
        ("dataset_analysis", analysis_count, original_count - analysis_count)
    ],
    ["stage", "total_rows", "rows_removed_vs_original"]
)

crud_d1_delete_summary.show(truncate=False)

+----------------+----------+------------------------+
|stage           |total_rows|rows_removed_vs_original|
+----------------+----------+------------------------+
|dataset_original|296740    |0                       |
|dataset_clean   |293267    |3473                    |
|dataset_analysis|279493    |17247                   |
+----------------+----------+------------------------+



## SPARK-R2: Read — resumen por tipo de cocina

In [31]:
crud_r2_cuisine_summary = (
    df_analysis.groupBy("cuisine_description")
    .agg(
        countDistinct("inspection_id").alias("total_inspections"),
        countDistinct("camis").alias("total_restaurants"),
        round(avg("score"), 2).alias("avg_score"),
        spark_sum("is_critical").alias("critical_violations")
    )
    .withColumn(
        "critical_rate",
        round((col("critical_violations") / col("total_inspections")) * 100, 2)
    )
    .orderBy(desc("critical_rate"))
)

crud_r2_cuisine_summary.show(20, truncate=False)

+-------------------+-----------------+-----------------+---------+-------------------+-------------+
|cuisine_description|total_inspections|total_restaurants|avg_score|critical_violations|critical_rate|
+-------------------+-----------------+-----------------+---------+-------------------+-------------+
|Bangladeshi        |324              |77               |44.51    |1012               |312.35       |
|Chilean            |6                |2                |28.34    |17                 |283.33       |
|NULL               |16               |3                |34.55    |43                 |268.75       |
|Creole             |97               |21               |30.23    |255                |262.89       |
|Chinese/Japanese   |121              |36               |31.71    |316                |261.16       |
|Filipino           |144              |41               |34.83    |370                |256.94       |
|Indian             |1035             |321              |33.1     |2603           

# KPIs con Spark

Antes de los KPIs, creamos una tabla a nivel de inspección. Esto corrige el problema de contar filas como inspecciones.

In [32]:
inspection_level = (
    df_analysis.groupBy(
        "camis",
        "dba",
        "boro_corrected",
        "cuisine_description",
        "inspection_date",
        "inspection_type",
        "inspection_id",
        "year",
        "month",
        "season"
    )
    .agg(
        round(avg("score"), 2).alias("inspection_score"),
        spark_sum("is_critical").alias("critical_violations"),
        spark_sum("has_violation").alias("total_violations")
    )
)

## KPI 1: Índice de efectividad de reinspección

In [33]:
window_restaurant = Window.partitionBy("camis").orderBy("inspection_date", "inspection_id")

df_kpi1 = (
    inspection_level
    .withColumn("previous_score", lag("inspection_score").over(window_restaurant))
    .withColumn("previous_inspection_date", lag("inspection_date").over(window_restaurant))
)

In [34]:
df_kpi1 = df_kpi1.filter(
    (
        lower(col("inspection_type")).contains("re-inspection") |
        lower(col("inspection_type")).contains("reinspection")
    ) &
    col("previous_score").isNotNull() &
    col("inspection_score").isNotNull() &
    (col("previous_score") > 0)
)

In [35]:
df_kpi1 = (
    df_kpi1
    .withColumn(
        "reinspection_effectiveness",
        round(
            ((col("previous_score") - col("inspection_score")) / col("previous_score")) * 100,
            2
        )
    )
    .withColumn(
        "result",
        when(col("reinspection_effectiveness") > 0, "Mejoró")
        .when(col("reinspection_effectiveness") == 0, "Sin cambio")
        .otherwise("Empeoró")
    )
)

kpi_1_summary = df_kpi1.agg(
    count("*").alias("total_reinspections_analyzed"),
    round(avg("reinspection_effectiveness"), 2).alias("avg_reinspection_effectiveness")
)

kpi_1_summary.show(truncate=False)

df_kpi1.select(
    "camis",
    "dba",
    "boro_corrected",
    "previous_inspection_date",
    "inspection_date",
    "previous_score",
    "inspection_score",
    "reinspection_effectiveness",
    "result"
).show(20, truncate=False)

+----------------------------+------------------------------+
|total_reinspections_analyzed|avg_reinspection_effectiveness|
+----------------------------+------------------------------+
|18938                       |30.1                          |
+----------------------------+------------------------------+



+--------+----------------------------------+--------------+------------------------+-------------------+--------------+----------------+--------------------------+-------+
|camis   |dba                               |boro_corrected|previous_inspection_date|inspection_date    |previous_score|inspection_score|reinspection_effectiveness|result |
+--------+----------------------------------+--------------+------------------------+-------------------+--------------+----------------+--------------------------+-------+
|30075445|MORRIS PARK BAKE SHOP             |Bronx         |2023-08-01 00:00:00     |2023-08-22 00:00:00|38.0          |12.0            |68.42                     |Mejoró |
|30191841|D.J. REYNOLDS                     |Manhattan     |2024-11-20 00:00:00     |2025-02-20 00:00:00|24.0          |10.0            |58.33                     |Mejoró |
|40363298|CAFE METRO                        |Manhattan     |2022-08-19 00:00:00     |2022-09-02 00:00:00|26.0          |2.0            

## KPI 2: Índice de deterioro estacional

In [36]:
season_summary = (
    inspection_level
    .filter(col("season").isin("Verano", "Invierno"))
    .groupBy("boro_corrected", "cuisine_description", "season")
    .agg(
        round(avg("inspection_score"), 2).alias("avg_score"),
        countDistinct("inspection_id").alias("total_inspections"),
        spark_sum("critical_violations").alias("critical_violations")
    )
)

In [37]:
summer = season_summary.filter(col("season") == "Verano").select(
    "boro_corrected",
    "cuisine_description",
    col("avg_score").alias("summer_avg_score"),
    col("total_inspections").alias("summer_inspections"),
    col("critical_violations").alias("summer_critical_violations")
)

winter = season_summary.filter(col("season") == "Invierno").select(
    "boro_corrected",
    "cuisine_description",
    col("avg_score").alias("winter_avg_score"),
    col("total_inspections").alias("winter_inspections"),
    col("critical_violations").alias("winter_critical_violations")
)

In [38]:
kpi_2 = (
    summer.join(
        winter,
        on=["boro_corrected", "cuisine_description"],
        how="inner"
    )
    .filter(col("winter_avg_score") > 0)
    .withColumn(
        "seasonal_deterioration_index",
        round(
            ((col("summer_avg_score") - col("winter_avg_score")) / col("winter_avg_score")) * 100,
            2
        )
    )
    .orderBy(desc("seasonal_deterioration_index"))
)

kpi_2.show(30, truncate=False)

+--------------+-------------------------+----------------+------------------+--------------------------+----------------+------------------+--------------------------+----------------------------+
|boro_corrected|cuisine_description      |summer_avg_score|summer_inspections|summer_critical_violations|winter_avg_score|winter_inspections|winter_critical_violations|seasonal_deterioration_index|
+--------------+-------------------------+----------------+------------------+--------------------------+----------------+------------------+--------------------------+----------------------------+
|Staten Island |Caribbean                |10.5            |2                 |2                         |2.0             |1                 |0                         |425.0                       |
|Manhattan     |Basque                   |41.0            |1                 |5                         |9.0             |1                 |1                         |355.56                      |
|Queens   

## KPI 5: Índice de riesgo sanitario por zona geográfica

In [39]:
zone_base = (
    inspection_level.groupBy("boro_corrected")
    .agg(
        countDistinct("inspection_id").alias("total_inspections"),
        countDistinct("camis").alias("total_restaurants"),
        round(avg("inspection_score"), 2).alias("avg_score"),
        spark_sum("critical_violations").alias("critical_violations")
    )
)

In [40]:
restaurant_recurrence = (
    inspection_level.groupBy("boro_corrected", "camis")
    .agg(
        spark_sum("total_violations").alias("total_violations_restaurant")
    )
    .withColumn(
        "is_recurrent",
        when(col("total_violations_restaurant") > 1, 1).otherwise(0)
    )
)

recurrence_by_boro = (
    restaurant_recurrence.groupBy("boro_corrected")
    .agg(
        spark_sum("is_recurrent").alias("recurrent_restaurants")
    )
)

In [41]:
kpi_5 = (
    zone_base.join(recurrence_by_boro, on="boro_corrected", how="left")
    .withColumn(
        "critical_rate",
        round((col("critical_violations") / col("total_inspections")) * 100, 2)
    )
    .withColumn(
        "recurrence_rate",
        round((col("recurrent_restaurants") / col("total_restaurants")) * 100, 2)
    )
    .withColumn(
        "sanitary_risk_index",
        round(col("avg_score") + col("critical_rate") + col("recurrence_rate"), 2)
    )
    .orderBy(desc("sanitary_risk_index"))
)

kpi_5.show(truncate=False)

+--------------+-----------------+-----------------+---------+-------------------+---------------------+-------------+---------------+-------------------+
|boro_corrected|total_inspections|total_restaurants|avg_score|critical_violations|recurrent_restaurants|critical_rate|recurrence_rate|sanitary_risk_index|
+--------------+-----------------+-----------------+---------+-------------------+---------------------+-------------+---------------+-------------------+
|Queens        |19508            |6288             |19.06    |39316              |6157                 |201.54       |97.92          |318.52             |
|Bronx         |7691             |2412             |17.46    |14234              |2346                 |185.07       |97.26          |299.79             |
|Brooklyn      |21760            |7029             |17.82    |39984              |6834                 |183.75       |97.23          |298.8              |
|Staten Island |3001             |1006             |16.95    |5468    

## KPI 6: Índice de excelencia sostenida

In [42]:
max_date = inspection_level.agg(
    spark_max("inspection_date").alias("max_date")
).collect()[0]["max_date"]

max_date

datetime.datetime(2026, 4, 28, 0, 0)

In [43]:
df_last_2_years = inspection_level.filter(
    col("inspection_date") >= add_months(lit(max_date), -24)
)

In [44]:
restaurant_2y = (
    df_last_2_years.groupBy(
        "camis", "dba", "boro_corrected", "cuisine_description"
    )
    .agg(
        countDistinct("inspection_id").alias("total_inspections_2y"),
        spark_sum("critical_violations").alias("critical_violations_2y")
    )
    .filter(col("total_inspections_2y") >= 3)
    .withColumn(
        "is_sustained_excellence",
        when(col("critical_violations_2y") == 0, 1).otherwise(0)
    )
)

In [45]:
kpi_6_summary = (
    restaurant_2y.agg(
        count("*").alias("restaurants_with_3_or_more_inspections"),
        spark_sum("is_sustained_excellence").alias("excellent_restaurants")
    )
    .withColumn(
        "sustained_excellence_index",
        round(
            (col("excellent_restaurants") / col("restaurants_with_3_or_more_inspections")) * 100,
            2
        )
    )
)

kpi_6_summary.show(truncate=False)

+--------------------------------------+---------------------+--------------------------+
|restaurants_with_3_or_more_inspections|excellent_restaurants|sustained_excellence_index|
+--------------------------------------+---------------------+--------------------------+
|5193                                  |16                   |0.31                      |
+--------------------------------------+---------------------+--------------------------+



In [46]:
kpi_6_by_boro = (
    restaurant_2y.groupBy("boro_corrected")
    .agg(
        count("*").alias("restaurants_with_3_or_more_inspections"),
        spark_sum("is_sustained_excellence").alias("excellent_restaurants")
    )
    .withColumn(
        "sustained_excellence_index",
        round(
            (col("excellent_restaurants") / col("restaurants_with_3_or_more_inspections")) * 100,
            2
        )
    )
    .orderBy(desc("sustained_excellence_index"))
)

kpi_6_by_boro.show(truncate=False)

+--------------+--------------------------------------+---------------------+--------------------------+
|boro_corrected|restaurants_with_3_or_more_inspections|excellent_restaurants|sustained_excellence_index|
+--------------+--------------------------------------+---------------------+--------------------------+
|Brooklyn      |1352                                  |7                    |0.52                      |
|Manhattan     |1785                                  |6                    |0.34                      |
|Queens        |1336                                  |3                    |0.22                      |
|Bronx         |588                                   |0                    |0.0                       |
|Staten Island |132                                   |0                    |0.0                       |
+--------------+--------------------------------------+---------------------+--------------------------+



# Guardado de resultados Spark en GCS

In [47]:
def write_csv_spark(df_spark, path):
    (
        df_spark
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", True)
        .csv(path)
    )
    print(f"Guardado en: {path}")

### Guardar CRUD

In [48]:
write_csv_spark(
    crud_c1_inspection_result.limit(1000),
    GCS_OUTPUTS_SPARK + "spark_crud_c1_inspection_result/"
)

write_csv_spark(
    crud_r1_boro_summary,
    GCS_OUTPUTS_SPARK + "spark_crud_r1_boro_summary/"
)

write_csv_spark(
    crud_u1_priority_attention.limit(1000),
    GCS_OUTPUTS_SPARK + "spark_crud_u1_priority_attention/"
)

write_csv_spark(
    crud_d1_delete_summary,
    GCS_OUTPUTS_SPARK + "spark_crud_d1_delete_summary/"
)

write_csv_spark(
    crud_r2_cuisine_summary,
    GCS_OUTPUTS_SPARK + "spark_crud_r2_cuisine_summary/"
)

Guardado en: gs://proyectobigdata2026/results/outputs/spark/spark_crud_c1_inspection_result/


Guardado en: gs://proyectobigdata2026/results/outputs/spark/spark_crud_r1_boro_summary/


Guardado en: gs://proyectobigdata2026/results/outputs/spark/spark_crud_u1_priority_attention/


Guardado en: gs://proyectobigdata2026/results/outputs/spark/spark_crud_d1_delete_summary/


Guardado en: gs://proyectobigdata2026/results/outputs/spark/spark_crud_r2_cuisine_summary/


### Guardar KPIs

In [49]:
write_csv_spark(
    kpi_1_summary,
    GCS_KPIS_SPARK + "spark_kpi1_reinspection_effectiveness_summary/"
)

write_csv_spark(
    df_kpi1.select(
        "camis",
        "dba",
        "boro_corrected",
        "previous_inspection_date",
        "inspection_date",
        "previous_score",
        "inspection_score",
        "reinspection_effectiveness",
        "result"
    ),
    GCS_KPIS_SPARK + "spark_kpi1_reinspection_effectiveness_detail/"
)

write_csv_spark(
    kpi_2,
    GCS_KPIS_SPARK + "spark_kpi2_seasonal_deterioration/"
)

write_csv_spark(
    kpi_5,
    GCS_KPIS_SPARK + "spark_kpi5_sanitary_risk_zone/"
)

write_csv_spark(
    kpi_6_summary,
    GCS_KPIS_SPARK + "spark_kpi6_sustained_excellence_summary/"
)

write_csv_spark(
    kpi_6_by_boro,
    GCS_KPIS_SPARK + "spark_kpi6_sustained_excellence_by_boro/"
)

Guardado en: gs://proyectobigdata2026/results/kpis/spark/spark_kpi1_reinspection_effectiveness_summary/


Guardado en: gs://proyectobigdata2026/results/kpis/spark/spark_kpi1_reinspection_effectiveness_detail/


Guardado en: gs://proyectobigdata2026/results/kpis/spark/spark_kpi2_seasonal_deterioration/


Guardado en: gs://proyectobigdata2026/results/kpis/spark/spark_kpi5_sanitary_risk_zone/


Guardado en: gs://proyectobigdata2026/results/kpis/spark/spark_kpi6_sustained_excellence_summary/


Guardado en: gs://proyectobigdata2026/results/kpis/spark/spark_kpi6_sustained_excellence_by_boro/


In [50]:
!gcloud storage ls gs://proyectobigdata2026/processed/spark/
!gcloud storage ls gs://proyectobigdata2026/results/outputs/spark/
!gcloud storage ls gs://proyectobigdata2026/results/kpis/spark/

gs://proyectobigdata2026/processed/spark/

gs://proyectobigdata2026/processed/spark/:
gs://proyectobigdata2026/processed/spark/
gs://proyectobigdata2026/processed/spark/restaurant_inspections_clean/
gs://proyectobigdata2026/results/outputs/spark/

gs://proyectobigdata2026/results/outputs/spark/:
gs://proyectobigdata2026/results/outputs/spark/
gs://proyectobigdata2026/results/outputs/spark/spark_crud_c1_inspection_result/
gs://proyectobigdata2026/results/outputs/spark/spark_crud_d1_delete_summary/
gs://proyectobigdata2026/results/outputs/spark/spark_crud_r1_boro_summary/
gs://proyectobigdata2026/results/outputs/spark/spark_crud_r2_cuisine_summary/
gs://proyectobigdata2026/results/outputs/spark/spark_crud_u1_priority_attention/
gs://proyectobigdata2026/results/kpis/spark/

gs://proyectobigdata2026/results/kpis/spark/:
gs://proyectobigdata2026/results/kpis/spark/
gs://proyectobigdata2026/results/kpis/spark/spark_kpi1_reinspection_effectiveness_detail/
gs://proyectobigdata2026/results/kpis

In [52]:
import psutil

mem = psutil.virtual_memory()

print(f"Total: {mem.total / 1e9:.2f} GB")
print(f"Usada: {mem.used / 1e9:.2f} GB")
print(f"Disponible: {mem.available / 1e9:.2f} GB")
print(f"Porcentaje: {mem.percent}%")

Total: 16.78 GB
Usada: 16.58 GB
Disponible: 0.20 GB
Porcentaje: 98.8%
